# 00 — SensoFit Data Release Walkthrough

This notebook accompanies the SensoFit data release distributed via Fragalysis.
It walks through, end-to-end, **how the published kinetic parameters were
computed** from the raw GCI sensorgrams, and explains the methodological
choices made along the way.

**Contents**

1. [Loading the data package](#1-loading-the-data-package)
2. [Non-convex optimisation: DK seeds vs random starting points](#2-non-convex-optimisation-dk-seeds-vs-random-starting-points)
3. [Residual weighting and goodness-of-fit](#3-residual-weighting-and-goodness-of-fit)

The notebook is self-contained: every helper function used in this analysis is
defined inline, so you can read it top-to-bottom without having to jump into
the `sensofit` source tree. Re-running it on the data package you downloaded
should reproduce the published numbers (modulo deterministic random seeds).

## 1. Loading the data package

The release ships as a `.zip` data package produced by
`sensofit.dataexporter.export_package`. Its layout is:

```
{package}.zip
├── README.md
└── {experiment}/
    ├── experiment.json
    ├── kinetics.csv
    └── {compound}__{conc}__cyc{idx:03d}/
        ├── metadata.json
        ├── FC2-FC1.csv      # time_s, signal, raw_active, raw_reference
        └── FC3-FC1.csv
```

`sensofit.load_experiment(path)` is a dispatcher that accepts:

- a `.cxw` file              → reads via `sensofit.load_cxw`
- a `.zip` data package      → reads via `sensofit.load_package`
- an unzipped package folder → reads via `sensofit.load_package`

The returned dict has identical shape in all three cases, so the rest of this
notebook (and indeed the rest of the `sensofit` API) works unchanged.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

from sensofit import load_experiment
from sensofit.models import (
    build_pulsed_concentration_profile,
    select_dmso_cal,
    build_full_weight_mask,
    trim_to_fit_window,
    simulate_sensorgram,
    is_nonspecific_binder,
)
from sensofit.direct_kinetics import fit_sample as dk_fit_sample
from sensofit.ode_fitting import fit_sample as ode_fit_sample

# --- Reusable loader cell --------------------------------------------------
# Edit INPUT to point at the data package you downloaded from Fragalysis.
INPUT = '../20260323_EV71 2A Binding assay.cxw'
# INPUT = '../20260323_EV71 2A Binding assay.cxw'   # raw .cxw fallback

data = load_experiment(INPUT)
samples      = data['samples']
blanks       = data['blanks']
dmso_cals    = data['dmso_cals']
other_cycles = data.get('other_cycles', [])

def find_sample(name, conc_uM=None):
    """Find a sample by compound name (substring match), optionally by concentration.
    If multiple matches, prompts the user to pick one."""
    hits = [(i, s) for i, s in enumerate(samples) if name in s['compound']]
    if conc_uM is not None:
        hits = [(i, s) for i, s in hits if abs(s['concentration_M'] * 1e6 - conc_uM) < 0.01]
    if not hits:
        raise ValueError(f'No sample found matching "{name}"' + (f' at {conc_uM} µM' if conc_uM else ''))
    if len(hits) == 1:
        return hits[0][1]
    print(f'Multiple matches for "{name}":')
    for j, (idx, s) in enumerate(hits):
        nsb, _ = is_nonspecific_binder(s)
        tag = ' [NSB]' if nsb else ''
        print(f'  [{j}] samples[{idx}]  {s["compound"]}  {s["concentration_M"]*1e6:.0f} µM  (cycle {s["index"]}){tag}')
    choice = int(input(f'Select [0–{len(hits)-1}]: '))
    return hits[choice][1]

print(f'Source:        {INPUT}')
print(f'Samples:       {len(samples)}')
print(f'Blanks:        {len(blanks)}')
print(f'DMSO Cals:     {len(dmso_cals)}')
print(f'Other cycles:  {len(other_cycles)}')

In [ ]:
# Quick inventory of the first 10 sample cycles
inv = pd.DataFrame([
    {'cycle_index': s['index'],
     'compound': s['compound'],
     'concentration_uM': s['concentration_M'] * 1e6,
     'mw': s.get('mw', 0)}
    for s in samples
])
inv.head(10)

In [ ]:
# Pick the featured sample used throughout this walkthrough.
# For interactive disambiguation between channels you can also call
# `find_sample('ASAP-0044081')`; here we pick the first match deterministically
# so the notebook can be re-run non-interactively.
sample = next(s for s in samples if 'ASAP-0044081' in s['compound'])
print(f'Featured sample: {sample["compound"]}  '
      f'({sample["concentration_M"]*1e6:.1f} µM, cycle {sample["index"]})')

## 2. Non-convex optimisation: DK seeds vs random starting points

The 1:1 Langmuir model

$$\frac{dR}{dt} = k_a \, c(t) \, [R_{\max} - R(t)] - k_d \, R(t)$$

is **non-linear** in its parameters $(k_a, k_d, R_{\max})$. The least-squares
loss surface has a long, curved valley in the $(k_a, R_{\max})$ plane (their
product sets the steady-state response) plus shallow local minima where the
optimiser can stall. As a result, the kinetic estimates returned by an ODE
solver are sensitive to where the optimiser is started.

`sensofit` defaults to **DK-seeded** starting points: the closed-form Direct
Kinetics solver provides initial guesses of $(k_a, k_d, R_{\max})$, and the
multi-start option then perturbs them log-normally
($\sigma = 0.5$ in log-space). This is fast and usually well-behaved.

To make the non-convexity visible, this section also runs the same fit from
**fully random log-uniform** starting points across the physical bounds used
internally by `ode_fit`:

| parameter | lower bound | upper bound |
|-----------|------------:|------------:|
| $k_a$       | $10^{-1}$ M⁻¹ s⁻¹ | $10^{8}$ M⁻¹ s⁻¹ |
| $k_d$       | $10^{-6}$ s⁻¹     | $10^{1}$ s⁻¹     |
| $R_{\max}$  | $1$ pg/mm²        | $10^{4}$ pg/mm²  |

We compare the resulting probability distributions of $k_a$, $k_d$, $R_{\max}$
and $K_D$, and show why the **median** (not the mean) is the right summary
statistic when distributions are heavy-tailed.

In [ ]:
# --- Replicate the prep that ode_fit_sample does internally ----------------
# (DK initial estimates, pulsed c(t), full weight mask, trimmed window)
dk = dk_fit_sample(sample, dmso_cals, blanks=blanks)
t_full, signal_full = dk['t'], dk['signal']

dmso = select_dmso_cal(sample['index'], dmso_cals)
c_func, _ = build_pulsed_concentration_profile(dmso, sample['concentration_M'])

# Default association_weight=0 → fit on buffer pulses + dissociation only
w_full = build_full_weight_mask(t_full, sample['markers'], dmso, association_weight=0.0)
t_fit, sig_fit, w_fit, fit_mask = trim_to_fit_window(
    t_full, signal_full, w_full, sample['markers'])

print(f'DK initial estimates:')
print(f'  ka   = {dk["ka"]:.2e} M⁻¹ s⁻¹')
print(f'  kd   = {dk["kd"]:.4f} s⁻¹')
print(f'  Rmax = {dk["Rmax"]:.1f} pg/mm²')
print(f'  KD   = {dk["KD"]*1e6:.2f} µM')
print(f'\nFitting window: {len(t_fit)} pts, {w_fit.sum():.0f} weighted')

In [ ]:
# --- Inline residual function (4-line wrapper around simulate_sensorgram) --
def _residuals(params, t, signal, c_func, w):
    """Weighted residuals for the 1:1 Langmuir ODE."""
    ka, kd, Rmax = params
    R_sim = simulate_sensorgram(t, ka, kd, Rmax, c_func, R0=0.0)
    return w * (signal - R_sim)

# ode_fit's physical bounds
LB = np.array([1e-1, 1e-6, 1.0])     # ka, kd, Rmax
UB = np.array([1e8,  1e1,  1e4])

def random_log_uniform_starts(n, rng):
    """Draw `n` random (ka, kd, Rmax) starts log-uniformly within [LB, UB]."""
    log_lo, log_hi = np.log10(LB), np.log10(UB)
    return [10 ** rng.uniform(log_lo, log_hi) for _ in range(n)]

def dk_perturbed_starts(n, rng, dk_seed, sigma=0.5):
    """Draw `n` log-normally perturbed starts around the DK seed (sensofit default)."""
    seed = np.array(dk_seed, dtype=float)
    return [np.clip(seed * np.exp(rng.normal(0, sigma, size=3)), LB, UB)
            for _ in range(n)]

def run_starts(starts, t, signal, c_func, w):
    """Run scipy.least_squares from each start; return tidy DataFrame."""
    rows = []
    for p0 in starts:
        try:
            opt = least_squares(
                _residuals, p0, args=(t, signal, c_func, w),
                bounds=(LB, UB), method='trf',
                ftol=1e-6, xtol=1e-6, gtol=1e-6, max_nfev=200, diff_step=1e-2)
            rows.append({
                'init_ka': p0[0], 'init_kd': p0[1], 'init_Rmax': p0[2],
                'ka': opt.x[0], 'kd': opt.x[1], 'Rmax': opt.x[2],
                'KD': opt.x[1] / opt.x[0],
                'cost': float(opt.cost), 'success': bool(opt.success),
                'nfev': int(opt.nfev),
            })
        except Exception:
            rows.append({
                'init_ka': p0[0], 'init_kd': p0[1], 'init_Rmax': p0[2],
                'ka': np.nan, 'kd': np.nan, 'Rmax': np.nan, 'KD': np.nan,
                'cost': np.nan, 'success': False, 'nfev': 0,
            })
    return pd.DataFrame(rows)

In [ ]:
# --- Run both strategies (this takes ~30–90 s) -----------------------------
N_STARTS = 5
RNG_SEED = 20260424
rng = np.random.default_rng(RNG_SEED)

dk_seed = (dk['ka'], dk['kd'], dk['Rmax'])
dk_results   = run_starts(dk_perturbed_starts(N_STARTS, rng, dk_seed),
                          t_fit, sig_fit, c_func, w_fit)
rand_results = run_starts(random_log_uniform_starts(N_STARTS, rng),
                          t_fit, sig_fit, c_func, w_fit)

dk_ok   = dk_results[dk_results['success']]
rand_ok = rand_results[rand_results['success']]
print(f'DK-perturbed:  {len(dk_ok)}/{N_STARTS} converged')
print(f'Random log-U:  {len(rand_ok)}/{N_STARTS} converged')

In [ ]:
# --- Histograms of ka, kd, Rmax, KD with median (solid) and mean (dashed) --
published = ode_fit_sample(sample, dmso_cals, blanks=blanks, n_starts=1)

params = [('ka', 'k_a (M⁻¹ s⁻¹)', True),
          ('kd', 'k_d (s⁻¹)',     True),
          ('Rmax', 'R_max (pg/mm²)', False),
          ('KD', 'K_D (M)',        True)]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (col, label, log_x) in zip(axes.flat, params):
    dk_vals   = dk_ok[col].dropna().values
    rand_vals = rand_ok[col].dropna().values

    if log_x:
        bins = np.logspace(np.log10(max(min(dk_vals.min(),  rand_vals.min()), 1e-12)),
                           np.log10(max(dk_vals.max(),     rand_vals.max())), 40)
        ax.set_xscale('log')
    else:
        bins = np.linspace(min(dk_vals.min(),  rand_vals.min()),
                           max(dk_vals.max(),  rand_vals.max()), 40)

    ax.hist(dk_vals,   bins=bins, alpha=0.55, color='steelblue',
            label=f'DK-perturbed (n={len(dk_vals)})')
    ax.hist(rand_vals, bins=bins, alpha=0.55, color='darkorange',
            label=f'Random log-U (n={len(rand_vals)})')

    for vals, color in [(dk_vals, 'steelblue'), (rand_vals, 'darkorange')]:
        ax.axvline(np.median(vals), color=color, lw=1.6, ls='-')
        ax.axvline(np.mean(vals),   color=color, lw=1.2, ls='--')

    ax.axvline(published[col], color='black', lw=1.2, ls=':',
               label=f'Published (single start)')
    ax.set_xlabel(label)
    ax.set_ylabel('count')
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

fig.suptitle(f'Parameter distributions across {N_STARTS} starts — '
             f'{sample["compound"]} ({sample["concentration_M"]*1e6:.1f} µM)\n'
             'Solid line = median, dashed line = mean, dotted = published',
             fontsize=11)
fig.tight_layout()
plt.show()

In [ ]:
# --- Tabulate median / mean / IQR / convergence rate -----------------------
def _summary(df, name):
    out = {'strategy': name, 'converged_pct': 100 * df['success'].mean()}
    ok = df[df['success']]
    for col in ['ka', 'kd', 'Rmax', 'KD']:
        v = ok[col].dropna().values
        if len(v):
            q25, q75 = np.percentile(v, [25, 75])
            out[f'{col}_median'] = np.median(v)
            out[f'{col}_mean']   = np.mean(v)
            out[f'{col}_iqr']    = q75 - q25
    return out

summary = pd.DataFrame([
    _summary(dk_results,   'DK-perturbed'),
    _summary(rand_results, 'Random log-U'),
    {'strategy': 'Published (1 start)', 'converged_pct': 100.0,
     'ka_median': published['ka'],     'ka_mean': published['ka'],     'ka_iqr': 0,
     'kd_median': published['kd'],     'kd_mean': published['kd'],     'kd_iqr': 0,
     'Rmax_median': published['Rmax'], 'Rmax_mean': published['Rmax'], 'Rmax_iqr': 0,
     'KD_median':   published['KD'],   'KD_mean':   published['KD'],   'KD_iqr':   0},
])
summary.set_index('strategy').T

**Discussion.** A few things to take away from the histograms above:

1. **Random starts find more local minima.** The orange (random log-uniform)
   distributions are visibly broader and often bimodal — the optimiser ends
   up in a shallow local minimum from a poor starting point. The blue
   (DK-perturbed) distributions are tighter because DK already sits in the
   right basin of attraction.
2. **The mean is misleading on heavy-tailed PDFs.** When even a handful of
   starts land in a bad local minimum (e.g.\ unrealistically large $k_a$
   compensated by tiny $R_{\max}$), the mean is pulled towards them. The
   median is robust.
3. **`sensofit` aggregates by median.** This is why `ode_fit` returns the
   median over converged multi-start fits, and why the published value
   (single-start, deterministic) sits very close to the **median** — not the
   mean — of the DK-perturbed distribution.

## 3. Residual weighting and goodness-of-fit

Pulsed waveform GCI sensorgrams contain three distinct regions: a baseline
before injection, an association phase made of alternating *analyte pulses*
(contaminated by RI bulk artefacts) and *buffer pulses* (clean binding
signal), and a dissociation phase after rinse-in. `sensofit` uses
`build_full_weight_mask` to weight residuals as

| region                     | default weight |
|----------------------------|---------------:|
| baseline                   | 0              |
| analyte pulses (RI)        | `association_weight` |
| buffer pulses              | 1              |
| dissociation (Rinse → end) | 1              |

The choice of `association_weight` ∈ [0, 1] is a trade-off:

- `0.0` (default) — fit on buffer pulses + dissociation only. Robust to RI
  artefacts but throws away kinetic information from the analyte pulses.
- `1.0` — fit everything. Squeezes all available data but is biased by RI
  spikes during the analyte pulses.

We sweep `association_weight` from 0 to 1 and report several
goodness-of-fit (GOF) metrics, **split by association vs dissociation
phase**, to show how the optimum shifts.

In [ ]:
# --- GOF helpers (chi^2, R^2, RMSE from cedric-dev; AIC/BIC added) --------
def _align(exp, fit):
    n = min(len(exp), len(fit))
    return np.asarray(exp[:n]), np.asarray(fit[:n])

def chi2(exp, fit, dof=3):
    exp, fit = _align(exp, fit)
    sigma = np.std(exp) if np.std(exp) > 0 else 1.0
    return float(np.sum(((exp - fit) / sigma) ** 2) / dof)

def r_squared(exp, fit):
    exp, fit = _align(exp, fit)
    ss_res = np.sum((exp - fit) ** 2)
    ss_tot = np.sum((exp - np.mean(exp)) ** 2)
    return float(1 - ss_res / ss_tot) if ss_tot > 0 else float('nan')

def rmse(exp, fit, p=3):
    exp, fit = _align(exp, fit)
    ss_res = np.sum((exp - fit) ** 2)
    return float(np.sqrt(ss_res / max(len(exp) - p, 1)))

def aic(exp, fit, k=3):
    exp, fit = _align(exp, fit)
    n = len(exp); rss = float(np.sum((exp - fit) ** 2))
    if rss <= 0 or n <= 0:
        return float('nan')
    return n * np.log(rss / n) + 2 * k

def bic(exp, fit, k=3):
    exp, fit = _align(exp, fit)
    n = len(exp); rss = float(np.sum((exp - fit) ** 2))
    if rss <= 0 or n <= 0:
        return float('nan')
    return n * np.log(rss / n) + k * np.log(n)

In [ ]:
# --- Sweep association_weight and collect kinetic params + per-phase GOF --
def weight_sweep(sample, dmso_cals, blanks, weights):
    rinse     = sample['markers'].get('Rinse',    0.0)
    rinse_end = sample['markers'].get('RinseEnd', np.inf)
    rows = []
    for w in weights:
        ode = ode_fit_sample(sample, dmso_cals, blanks=blanks,
                             association_weight=w, n_starts=1)
        t_o   = ode['t']
        sig_o = ode['signal']
        R_fit = ode['R_fit']
        valid = ~np.isnan(R_fit)
        diss  = valid & (t_o >= rinse) & (t_o <= rinse_end)
        asso  = valid & ~diss
        row = {
            'weight': w,
            'ka': ode['ka'], 'ka_se': ode.get('ka_se', np.nan),
            'kd': ode['kd'], 'kd_se': ode.get('kd_se', np.nan),
            'Rmax': ode['Rmax'], 'Rmax_se': ode.get('Rmax_se', np.nan),
            'KD': ode['KD'],
            'chi2_total': chi2(sig_o[valid], R_fit[valid]),
            'chi2_asso':  chi2(sig_o[asso],  R_fit[asso])  if asso.sum() else np.nan,
            'chi2_diss':  chi2(sig_o[diss],  R_fit[diss])  if diss.sum() else np.nan,
            'rmse_total': rmse(sig_o[valid], R_fit[valid]),
            'rmse_asso':  rmse(sig_o[asso],  R_fit[asso])  if asso.sum() else np.nan,
            'rmse_diss':  rmse(sig_o[diss],  R_fit[diss])  if diss.sum() else np.nan,
            'r2_total':   r_squared(sig_o[valid], R_fit[valid]),
            'aic':        aic(sig_o[valid], R_fit[valid]),
            'bic':        bic(sig_o[valid], R_fit[valid]),
        }
        rows.append(row)
    return pd.DataFrame(rows)

WEIGHTS = np.round(np.linspace(0, 1, 11), 2)
ws_df = weight_sweep(sample, dmso_cals, blanks, WEIGHTS)
ws_df.round(4)

In [ ]:
# --- Two-row figure: kinetic params (top) + per-phase GOF (bottom) --------
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Row 1 — parameters with SE error bars
for ax, col, ylabel, log_y in zip(
        axes[0],
        ['ka', 'kd', 'Rmax', 'KD'],
        ['k_a (M⁻¹ s⁻¹)', 'k_d (s⁻¹)', 'R_max (pg/mm²)', 'K_D (M)'],
        [True, True, False, True]):
    se_col = f'{col}_se' if f'{col}_se' in ws_df.columns else None
    yerr = ws_df[se_col] if se_col else None
    ax.errorbar(ws_df['weight'], ws_df[col], yerr=yerr,
                marker='o', color='steelblue', capsize=3)
    if log_y:
        ax.set_yscale('log')
    ax.set_xlabel('association_weight')
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)

# Row 2 — GOF metrics, per-phase
for ax, metric, ylabel in zip(
        axes[1],
        ['chi2', 'rmse', 'r2', 'aic'],
        ['χ² (per dof)', 'RMSE (pg/mm²)', 'R²', 'AIC']):
    if metric == 'r2':
        ax.plot(ws_df['weight'], ws_df['r2_total'], 'k-o', label='total')
    elif metric == 'aic':
        ax.plot(ws_df['weight'], ws_df['aic'], 'k-o', label='AIC')
        ax.plot(ws_df['weight'], ws_df['bic'], 'r-s', label='BIC')
    else:
        ax.plot(ws_df['weight'], ws_df[f'{metric}_total'], 'k-o', label='total')
        ax.plot(ws_df['weight'], ws_df[f'{metric}_asso'],  'b--^', label='association')
        ax.plot(ws_df['weight'], ws_df[f'{metric}_diss'],  'g--v', label='dissociation')
    ax.set_xlabel('association_weight')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(f'Weight sensitivity — {sample["compound"]} '
             f'({sample["concentration_M"]*1e6:.1f} µM)', fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# --- Heatmap of z-scored GOF metrics vs weight (matplotlib only) ---------
metric_cols = ['chi2_total', 'chi2_asso', 'chi2_diss',
               'rmse_total', 'rmse_asso', 'rmse_diss',
               'aic', 'bic']
heat = ws_df[metric_cols].copy()
# Z-score each metric so they share a colour scale (lower = better, dark = better)
heat_z = (heat - heat.mean()) / heat.std(ddof=0)
Z = heat_z.T.values            # rows = metrics, cols = weights
A = heat.T.round(2).values     # raw annotations

fig, ax = plt.subplots(figsize=(10, 5))
vmax = float(np.nanmax(np.abs(Z)))
im = ax.imshow(Z, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
ax.set_xticks(range(len(ws_df)))
ax.set_xticklabels([f'{w:.1f}' for w in ws_df['weight']])
ax.set_yticks(range(len(metric_cols)))
ax.set_yticklabels(metric_cols)

for i in range(Z.shape[0]):
    for j in range(Z.shape[1]):
        ax.text(j, i, f'{A[i, j]:g}', ha='center', va='center',
                fontsize=8,
                color='white' if abs(Z[i, j]) > 0.6 * vmax else 'black')
fig.colorbar(im, ax=ax, label='z-score (lower = better)')
ax.set_title(f'Goodness-of-fit vs association_weight — {sample["compound"]}')
ax.set_xlabel('association_weight')
ax.set_ylabel('metric')
fig.tight_layout()
plt.show()

**Discussion.** The sweep makes a few patterns clear:

- **Pure dissociation fitting (`weight = 0`)** has the lowest dissociation
  RMSE — by construction — but pays for it with a worse association RMSE
  and noisier $R_{\max}$ (because $R_{\max}$ is anchored only by where the
  curve plateaus during the buffer pulses).
- **Full-weight fitting (`weight = 1`)** achieves the best total χ²/RMSE
  but biases the parameters: the analyte-pulse RI spikes get folded into
  the residual and the optimiser distorts $k_a$ to absorb them.
- **The composite GOF (heatmap row "chi2_total" / "rmse_total")** typically
  reaches its minimum somewhere in **0.1–0.3**, balancing the two phases.
  AIC and BIC track RMSE here because $k = 3$ is fixed across the sweep, so
  they offer no extra parsimony information.

The published kinetics in this release use the `sensofit` defaults
(`association_weight = 0.0`, dissociation-only fitting), which is the most
robust choice when the goal is comparing many compounds across a screen.
For tighter mechanistic interpretation of a single sensorgram, raising
`association_weight` to ~0.2 is often worthwhile.

## Reproducibility

- Random seed for the multi-start study: `RNG_SEED = 20260424`
  (change this and re-run section 2 to convince yourself the broad shape
  of the histograms is seed-independent)
- All helper functions (`_residuals`, `random_log_uniform_starts`,
  `dk_perturbed_starts`, `run_starts`, the GOF metrics, and `weight_sweep`)
  are defined inline in this notebook — no `sensofit` private API is used.
- Source repository: <https://github.com/asapdiscovery/sensofit>